# Euribor Rate Explorer

**Data source:** [euriborrates.com](https://euriborrates.com/) — an independent, non-commercial, non-profit information resource.

This notebook fetches Euribor fixings (1W, 1M, 3M, 6M, 12M) and displays the current state of our dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import pandas as pd
import math
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

from pricebook.data.euribor_loader import (
    fetch_date, fetch_year_all_tenors, attribution, TENORS, TENOR_LABELS,
)
from pricebook.viz import configure_theme, correlation_heatmap, pnl_distribution
from pricebook.viz._theme import get_theme
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.interpolation import InterpolationMethod
from pricebook.core.day_count import DayCountConvention
from pricebook.curves.bootstrap import bootstrap
from pricebook.curves.nelson_siegel import calibrate_nelson_siegel, nelson_siegel_yield
import matplotlib.pyplot as plt

configure_theme()
theme = get_theme()
print(attribution())

## 1. Fetch Today's Fixing

In [ ]:
# Fetch today's rates (all 5 tenors)
today = date.today()
fixing = fetch_date(today)

if fixing:
    print(f"Euribor fixings for {fixing.date}")
    print(f"{'Tenor':<12} {'Rate':>8}")
    print("-" * 22)
    for tenor in TENORS:
        rate = fixing.rates.get(tenor)
        if rate is not None:
            print(f"{TENOR_LABELS[tenor]:<12} {rate*100:>7.3f}%")
else:
    print(f"No fixing available for {today} (weekend or holiday?)")

## 2. Fetch a Sample Year (all 5 tenors)

In [ ]:
# Fetch 2024 with all 5 tenors (5 requests, ~10s)
fixings_2024 = fetch_year_all_tenors(2024, delay_between_tenors=1.5)
print(f"Fetched {len(fixings_2024)} business days for 2024")

# Convert to DataFrame
rows = []
for f in fixings_2024:
    row = {"date": f.date}
    for t in TENORS:
        row[t] = f.rates.get(t)
    rows.append(row)

df_2024 = pd.DataFrame(rows).set_index("date")
df_2024.columns = [TENOR_LABELS[t] for t in TENORS]
df_2024 = df_2024 * 100  # percentage

# Helper: tenor offsets for curve building
TENOR_OFFSETS = {
    "1w": relativedelta(weeks=1),
    "1m": relativedelta(months=1),
    "3m": relativedelta(months=3),
    "6m": relativedelta(months=6),
    "12m": relativedelta(months=12),
}
TENOR_YEARS = {"1w": 7/365, "1m": 1/12, "3m": 0.25, "6m": 0.5, "12m": 1.0}

df_2024.head()

In [ ]:
# Last 5 days
df_2024.tail()

## 3. Bootstrap EUR Discount Curve

Build a `DiscountCurve` from Euribor deposits using pricebook's `bootstrap()` with EUR ACT/360 convention. Then extract discount factors, zero rates, and instantaneous forwards directly from the curve object.

In [ ]:
# Bootstrap curve from the last day of 2024
def build_curve_from_fixing(fixing):
    """Build a DiscountCurve from an EuriborFixing using pricebook bootstrap."""
    ref = fixing.date
    deposits = [(ref + TENOR_OFFSETS[t], fixing.rates[t]) for t in TENORS if t in fixing.rates]
    return bootstrap(ref, deposits, swaps=[], deposit_day_count=DayCountConvention.ACT_360,
                     interpolation=InterpolationMethod.LOG_LINEAR)

curve = build_curve_from_fixing(fixings_2024[-1])
ref_date = fixings_2024[-1].date

# Extract curve analytics using DiscountCurve methods
print(f"EUR Discount Curve — {ref_date}")
print(f"{'Tenor':<10} {'Maturity':<12} {'DF':>10} {'Zero Rate':>10} {'Fwd Rate':>10}")
print("-" * 56)
prev_date = ref_date
for tenor in TENORS:
    mat = ref_date + TENOR_OFFSETS[tenor]
    df_val = curve.df(mat)
    zr = curve.zero_rate(mat) * 100
    fwd = curve.forward_rate(prev_date, mat) * 100
    print(f"{TENOR_LABELS[tenor]:<10} {mat} {df_val:>10.6f} {zr:>9.3f}% {fwd:>9.3f}%")
    prev_date = mat

print(f"\nSource: euriborrates.com")

## 4. Discount Factor, Zero Rate & Forward Curves

All extracted from the bootstrapped `DiscountCurve` object.

In [ ]:
# Plot DF, zero rate, and instantaneous forward from the curve object
plot_dates = [ref_date + timedelta(days=d) for d in range(1, 370)]
days = [(d - ref_date).days for d in plot_dates]
dfs_plot = [curve.df(d) for d in plot_dates]
zeros_plot = [curve.zero_rate(d) * 100 for d in plot_dates]
fwds_plot = [curve.instantaneous_forward(d) * 100 for d in plot_dates]

# Pillar positions
pillar_days = [(ref_date + TENOR_OFFSETS[t] - ref_date).days for t in TENORS]
pillar_dfs = [curve.df(ref_date + TENOR_OFFSETS[t]) for t in TENORS]
pillar_zeros = [curve.zero_rate(ref_date + TENOR_OFFSETS[t]) * 100 for t in TENORS]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# DF
axes[0].plot(days, dfs_plot, color=theme.colors[0], linewidth=theme.line_width)
axes[0].scatter(pillar_days, pillar_dfs, color=theme.colors[1], zorder=5, s=50, label="Pillars")
axes[0].set_xlabel("Days"); axes[0].set_ylabel("DF")
axes[0].set_title("Discount Factors", fontsize=theme.title_size)
axes[0].legend(); axes[0].grid(True, alpha=theme.grid_alpha)

# Zero rate
axes[1].plot(days, zeros_plot, color=theme.colors[2], linewidth=theme.line_width)
axes[1].scatter(pillar_days, pillar_zeros, color=theme.colors[1], zorder=5, s=50)
axes[1].set_xlabel("Days"); axes[1].set_ylabel("Rate (%)")
axes[1].set_title("Zero Rates", fontsize=theme.title_size)
axes[1].grid(True, alpha=theme.grid_alpha)

# Instantaneous forward
axes[2].plot(days, fwds_plot, color=theme.colors[3], linewidth=theme.line_width)
axes[2].set_xlabel("Days"); axes[2].set_ylabel("Rate (%)")
axes[2].set_title("Inst. Forward", fontsize=theme.title_size)
axes[2].grid(True, alpha=theme.grid_alpha)

fig.suptitle(f"EUR Curve — {ref_date} (Source: euriborrates.com)", fontsize=theme.title_size, y=1.02)
plt.tight_layout()
plt.show()

## 5. Nelson-Siegel Calibration

Fit the Nelson-Siegel parametric model to the Euribor zero rates and compare with the bootstrapped curve.

In [ ]:
# Calibrate Nelson-Siegel to last-day zero rates
tenors_yr = [TENOR_YEARS[t] for t in TENORS]
market_zeros = [curve.zero_rate(ref_date + TENOR_OFFSETS[t]) for t in TENORS]

ns_params = calibrate_nelson_siegel(tenors_yr, market_zeros)
print("Nelson-Siegel calibration:")
for k, v in ns_params.items():
    print(f"  {k}: {v:.6f}")

# Compare NS fit vs bootstrap
ns_zeros = [nelson_siegel_yield(t, ns_params["beta0"], ns_params["beta1"],
                                  ns_params["beta2"], ns_params["tau"]) * 100
            for t in np.linspace(0.02, 1.0, 100)]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(np.linspace(0.02, 1.0, 100), ns_zeros, color=theme.colors[0],
        linewidth=theme.line_width, label="Nelson-Siegel fit")
ax.scatter(tenors_yr, [z * 100 for z in market_zeros], color=theme.colors[1],
           s=80, zorder=5, label="Bootstrap pillars")
ax.set_xlabel("Tenor (years)")
ax.set_ylabel("Zero Rate (%)")
ax.set_title(f"Nelson-Siegel vs Bootstrap — {ref_date}\n"
             f"β₀={ns_params['beta0']:.4f}, β₁={ns_params['beta1']:.4f}, "
             f"β₂={ns_params['beta2']:.4f}, τ={ns_params['tau']:.4f}, "
             f"RMSE={ns_params['rmse']*100:.4f}bp\n"
             f"Source: euriborrates.com", fontsize=theme.title_size - 1)
ax.legend()
ax.grid(True, alpha=theme.grid_alpha)
plt.tight_layout()
plt.show()

## 6. Tenor Correlation & Daily Change Distribution

Using pricebook's `correlation_heatmap` and `pnl_distribution` from `pricebook.viz`.

In [ ]:
# Correlation heatmap of daily rate changes (using pricebook.viz)
changes = df_2024.diff().dropna()
fig = correlation_heatmap(changes, title="Euribor Daily Change Correlation — 2024\nSource: euriborrates.com")
plt.show()

In [ ]:
# Distribution of 3M daily changes (using pricebook.viz)
changes_3m = df_2024["3 Months"].diff().dropna().values
fig = pnl_distribution(changes_3m, title="3M Euribor Daily Change Distribution — 2024\nSource: euriborrates.com")
plt.show()

## 7. Statistics & Curve Summary

In [ ]:
# Rate statistics
print("2024 Euribor Statistics (in %)")
print("=" * 55)
stats = df_2024.describe().T[["mean", "std", "min", "max"]]
stats.columns = ["Mean", "StdDev", "Min", "Max"]
print(stats.round(3).to_string())

# Curve bumped sensitivity
print(f"\n--- Curve Sensitivity (parallel +1bp) ---")
curve_bumped = curve.bumped(0.0001)
for tenor in TENORS:
    mat = ref_date + TENOR_OFFSETS[tenor]
    base_df = curve.df(mat)
    bump_df = curve_bumped.df(mat)
    sens = (bump_df - base_df) * 10_000  # in DV01 per 10k notional
    print(f"  {TENOR_LABELS[tenor]:<10}: DF change = {(bump_df-base_df):.8f}  (DV01/10k = {sens:.4f})")

print(f"\nBusiness days: {len(df_2024)}")
print(f"Date range: {df_2024.index[0]} to {df_2024.index[-1]}")
print(f"\nData source: euriborrates.com")

---

## Next Steps

1. **Full historical load** — fetch 1999–2026, all 5 tenors (~135 requests)
2. **Historical curve database** — bootstrap a `DiscountCurve` for every business day, store NS/Svensson parameters
3. **Daily cron** — automated daily update via `update_latest()`
4. **Analytics** — PCA on curve moves, regime detection, ECB impact, carry/roll-down backtest

*Data sourced from [euriborrates.com](https://euriborrates.com/) — an independent, non-commercial, non-profit information resource.*